![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20NB%20Header%20Banner.png)

# From Reviews to Results: An Interactive Introduction

Python is a powerful way to turn short pieces of text into useful evidence. In this notebook, we will work with a small review-style dataset and learn how to:

- use variables, strings, functions, and simple conditionals
- organise review text inside a pandas DataFrame
- clean text and create new features
- estimate simple sentiment with TextBlob
- build charts with Plotly
- create a striking word cloud
- explore the data with interactive widgets
- treat debugging as a normal and helpful part of programming

The goal is not just to make attractive outputs. It is to show how introductory programming helps us ask better questions, inspect data carefully, and explain what we find.

Even in a 2025 world full of AI tools, these core programming habits still matter: they help us understand where results come from, test ideas, and avoid treating text analysis like magic.

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the tools we need for text, data, visualisation, and interactivity. Click the code cell below and press the <b><code>▶</code> Run</b> button in the toolbar (or use <code>Shift + Enter</code>).
</div>

We will use a modern but beginner-friendly stack: pandas for tables, TextBlob for simple sentiment, Plotly for visualisation, WordCloud for a strong visual payoff, and ipywidgets for interactive exploration.

In [ ]:
print("Starting setup: importing libraries...")

import re
import string
import warnings
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
from textblob import TextBlob
from wordcloud import WordCloud

print("Success! All libraries imported. You are ready to analyse some text.")

## 2. Core Python Primer: Turning Text into Information

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Core idea:</b> A string is a piece of text, a variable stores a value, and a function lets us reuse code instead of rewriting the same steps again and again.
</div>

Before working with a full dataset, let's practise on one small review. This is a gentle way to see how Python handles text.

In [ ]:
print("Building a simple text-analysis function...")

# EDIT THIS VALUE: try changing the review text and run the cell again
sample_review = "The planner looks clean and the reminders are helpful!"

def describe_review(text):
    """Print a few beginner-friendly facts about a review."""
    word_count = len(text.split())
    exclamation_count = text.count("!")
    
    if "helpful" in text.lower() or "great" in text.lower() or "love" in text.lower():
        quick_tone = "This sounds positive."
    else:
        quick_tone = "This might be mixed or neutral."
    
    print(f"Original text: {text}")
    print(f"Uppercase version: {text.upper()}")
    print(f"Word count: {word_count}")
    print(f"Exclamation marks: {exclamation_count}")
    print(f"Quick interpretation: {quick_tone}")

describe_review(sample_review)
print("Function created and used successfully!")

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> Change the <code>sample_review</code> text in the cell above. Try a clearly positive sentence, then a clearly negative one. What changes? What stays the same?
</div>

## 3. Loading and Exploring the Data

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Load a compact review dataset, inspect it, and create useful new columns from the text. The dataset below is embedded directly in the notebook, so it is quick and reliable to run in class.
</div>

Our reviews are fictional but realistic. They represent short comments about student-facing services and tools.

In [ ]:
print("Creating a compact review dataset...")

review_data = [
    {"review_id": 1, "category": "Study App", "rating": 5, "review": "The weekly planner is brilliant. I finished my deadlines early and the reminders are so helpful!"},
    {"review_id": 2, "category": "Study App", "rating": 4, "review": "Clean design and quick sync across my phone and laptop. I wish the calendar colours were easier to edit."},
    {"review_id": 3, "category": "Study App", "rating": 2, "review": "The idea is good, but the app crashed twice during revision week and I lost my notes."},
    {"review_id": 4, "category": "Study App", "rating": 3, "review": "Useful for checklists, although some menus feel hidden and a bit confusing at first."},
    {"review_id": 5, "category": "Study App", "rating": 5, "review": "Love the focus timer! It helped me study in short bursts and the statistics made progress feel real."},
    {"review_id": 6, "category": "Study App", "rating": 1, "review": "Very frustrating. Notifications arrived late and the dashboard felt messy."},
    {"review_id": 7, "category": "Campus Café", "rating": 5, "review": "Great coffee, friendly staff, and the queue moved surprisingly fast even before my lecture."},
    {"review_id": 8, "category": "Campus Café", "rating": 2, "review": "The sandwich looked fresh, but it was expensive and the music was far too loud for studying."},
    {"review_id": 9, "category": "Campus Café", "rating": 4, "review": "Comfortable seats, plenty of plugs, and the hot chocolate was excellent."},
    {"review_id": 10, "category": "Campus Café", "rating": 3, "review": "Nice space overall, yet the tables were sticky when I arrived and service was slow."},
    {"review_id": 11, "category": "Campus Café", "rating": 5, "review": "Absolutely loved the soup today! Warm, quick, and perfect on a rainy afternoon."},
    {"review_id": 12, "category": "Campus Café", "rating": 1, "review": "I waited ages for tea and my order was incorrect."},
    {"review_id": 13, "category": "Streaming Service", "rating": 4, "review": "The recommendations were spot on and I discovered two new artists while writing an essay."},
    {"review_id": 14, "category": "Streaming Service", "rating": 2, "review": "Easy to use, but the adverts break my concentration and the search missed obvious results."},
    {"review_id": 15, "category": "Streaming Service", "rating": 5, "review": "Beautiful interface, smart playlists, and offline mode worked perfectly on the bus!"},
    {"review_id": 16, "category": "Streaming Service", "rating": 3, "review": "Decent sound quality, although some podcasts loaded slowly this morning."},
    {"review_id": 17, "category": "Streaming Service", "rating": 1, "review": "Terrible update. My downloads disappeared and the app froze after every track."},
    {"review_id": 18, "category": "Streaming Service", "rating": 4, "review": "Shared playlists made our group project session much more fun and focused."},
    {"review_id": 19, "category": "Library Space", "rating": 5, "review": "Quiet zone was peaceful, bright, and ideal for deep reading. I stayed productive for hours."},
    {"review_id": 20, "category": "Library Space", "rating": 2, "review": "Finding a seat was stressful and one area was much noisier than the signs suggested."},
    {"review_id": 21, "category": "Library Space", "rating": 4, "review": "Helpful staff, reliable Wi-Fi, and lots of power sockets near the windows."},
    {"review_id": 22, "category": "Library Space", "rating": 3, "review": "Solid study space, but it became crowded after lunch and the printers jammed."},
    {"review_id": 23, "category": "Library Space", "rating": 5, "review": "Booked room worked perfectly for group revision and the whiteboards were excellent."},
    {"review_id": 24, "category": "Library Space", "rating": 1, "review": "Air felt stuffy, the chairs were uncomfortable, and I left earlier than planned."}
]

reviews_df = pd.DataFrame(review_data)

print("Dataset ready! Here are the first few rows:")
display(reviews_df.head())

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why functions matter:</b> Once we have many reviews, it becomes inefficient to clean and analyse each one by hand. Reusable functions help us apply the same logic consistently to every row.
</div>

In the cell below, we will create several helpful features:

- <code>cleaned_review</code>: simplified text for word counting
- <code>char_count</code>: number of characters
- <code>word_count</code>: number of words
- <code>exclamation_count</code>: a small clue about emphasis
- <code>sentiment</code>: a simple polarity score from TextBlob
- <code>sentiment_label</code>: Positive, Neutral, or Negative

In [ ]:
print("Creating reusable helper functions for text processing...")

STOPWORDS = {
    "the", "and", "a", "an", "to", "of", "it", "is", "was", "are", "my", "i", "for", "on", "in",
    "but", "so", "at", "this", "that", "with", "me", "our", "very", "too", "than", "after"
}

def clean_text(text, remove_stopwords=True):
    """Lowercase text, remove punctuation, and optionally drop common stopwords."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    words = [word for word in text.split() if word]
    if remove_stopwords:
        words = [word for word in words if word not in STOPWORDS]
    return " ".join(words)

def get_word_count(text):
    return len(re.findall(r"\b\w+\b", text))

def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

def get_sentiment_label(polarity, positive_threshold=0.15, negative_threshold=-0.15):
    if polarity > positive_threshold:
        return "Positive"
    elif polarity < negative_threshold:
        return "Negative"
    else:
        return "Neutral"

def extract_top_words(text_series, top_n=10):
    all_words = " ".join(text_series.dropna()).split()
    counts = Counter(all_words)
    common_words = counts.most_common(top_n)
    return pd.DataFrame(common_words, columns=["word", "count"])

reviews_df["cleaned_review"] = reviews_df["review"].apply(clean_text)
reviews_df["char_count"] = reviews_df["review"].str.len()
reviews_df["word_count"] = reviews_df["review"].apply(get_word_count)
reviews_df["exclamation_count"] = reviews_df["review"].str.count("!")
reviews_df["sentiment"] = reviews_df["review"].apply(get_sentiment)
reviews_df["sentiment_label"] = reviews_df["sentiment"].apply(get_sentiment_label)

print("Text cleaned. Derived features added successfully!")
display(reviews_df[["category", "rating", "word_count", "exclamation_count", "sentiment", "sentiment_label"]].head())

In [ ]:
print("Inspecting the dataset...")

category_counts = reviews_df["category"].value_counts().rename_axis("category").reset_index(name="review_count")
category_summary = (
    reviews_df
    .groupby("category")
    .agg(
        review_count=("review", "size"),
        average_rating=("rating", "mean"),
        average_sentiment=("sentiment", "mean"),
        average_word_count=("word_count", "mean")
    )
    .round(2)
    .reset_index()
)

print("Reviews per category:")
display(category_counts)

print("Category-level summary:")
display(category_summary)

print("Most positive-sounding examples according to TextBlob:")
display(reviews_df.nlargest(3, "sentiment")[["category", "review", "sentiment"]])

## 4. Error Interpretation & Debugging

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Don't panic when you see red text!</b> Errors are part of programming, not proof that you are bad at it. This section is deliberately designed to show several common situations. If you are stepping through the notebook carefully, run these cells one at a time.
</div>

Just like in the template notebook, some cells below are meant to warn you or fail on purpose. That is the whole point of the exercise.

### A. Warnings (Useful Hints, Not Full Failures)

Sometimes Python gives you a heads-up about future changes. A warning is not necessarily fatal, and your cell will often still run.

In [ ]:
# This cell shows a warning, but it still completes.
warnings.warn(
    "FutureWarning: a later version of our text cleaner might use a different default stopword list.",
    FutureWarning
)

print("See? The code still ran after the warning.")

### B. Import Errors (Missing Tools)

If you try to import a package that is not installed in your environment, Python raises a <code>ModuleNotFoundError</code>.

<b>How to fix it:</b> Check the package name carefully, or install it if appropriate for your environment. In a managed teaching environment, it can also be worth checking the available notebook server preset first.

In [ ]:
print("Attempting to import a package that does not exist...")

# This will trigger an error on purpose.
import definitely_not_a_real_noteable_package

### C. Isolated Syntax Errors

A syntax error means Python cannot understand the shape of your code. This often happens because of a missing bracket, quote mark, or colon.

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Fix It:</b> Run the cell below, read the error carefully, then add the missing <code>)</code> and run it again.
</div>

In [ ]:
# Run this cell to see the SyntaxError, then fix it.
print("This review cell has a missing bracket"


### D. Cascading Errors

Sometimes one error stops a variable from ever being created. The next cell then fails too, not because it is wrong by itself, but because it depends on something that does not exist yet.

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Fix It:</b> In the first cell below, change <code>total_reviews = 0</code> to a sensible non-zero value such as <code>24</code>. Then rerun both cells in order.
</div>

In [ ]:
# This cell fails because division by zero is not allowed.
total_reviews = 0
positive_share = 12 / total_reviews

print("This line never runs because the error happens first.")

In [ ]:
# This cell depends on positive_share from the cell above.
percentage_value = positive_share * 100
print(f"Positive review share: {percentage_value:.1f}%")

## 5. Analysing and Visualising the Results

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Turn our cleaned text and derived features into visual evidence. We will summarise sentiment across categories, compare review length with polarity, and inspect common words.
</div>

These charts are not just decorative. They help us connect programming decisions to interpretable results.

In [ ]:
print("Generating category comparison chart...")

category_chart_df = (
    reviews_df
    .groupby("category")
    .agg(
        average_sentiment=("sentiment", "mean"),
        average_rating=("rating", "mean"),
        review_count=("review", "size")
    )
    .round(2)
    .reset_index()
)

fig = px.bar(
    category_chart_df,
    x="category",
    y="average_sentiment",
    color="average_rating",
    text="average_sentiment",
    color_continuous_scale="Viridis",
    template="plotly_white",
    title="Average sentiment by category"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(xaxis_title="", yaxis_title="Average sentiment polarity")
fig.show()

print("Chart generated successfully!")

In [ ]:
print("Plotting review length against sentiment...")

fig = px.scatter(
    reviews_df,
    x="word_count",
    y="sentiment",
    color="category",
    size="rating",
    hover_data=["review", "sentiment_label"],
    template="plotly_white",
    title="Do longer reviews sound more positive or negative?"
)

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(xaxis_title="Word count", yaxis_title="Sentiment polarity")
fig.show()

print("Scatter plot ready!")

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Word frequency:</b> After cleaning the text, we can count which words appear most often. This is a simple but useful bridge between raw language and structured analysis.
</div>

We will first make a ranked bar chart, then a word cloud for a more visual summary.

In [ ]:
print("Counting common words...")

# EDIT THIS VALUE: try 5, 10, or 15
top_n_words = 10

top_words_df = extract_top_words(reviews_df["cleaned_review"], top_n=top_n_words)

fig = px.bar(
    top_words_df.sort_values("count"),
    x="count",
    y="word",
    orientation="h",
    color="count",
    color_continuous_scale="Sunsetdark",
    template="plotly_white",
    title=f"Top {top_n_words} words across all reviews"
)

fig.update_layout(xaxis_title="Count", yaxis_title="")
fig.show()

print("Top-word chart generated successfully!")

In [ ]:
print("Generating word cloud...")

all_cleaned_text = " ".join(reviews_df["cleaned_review"])

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    colormap="magma"
).generate(all_cleaned_text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Cleaned Review Text", fontsize=16)
plt.show()

print("Word cloud complete!")

## 6. Interactive Explorations

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interactivity:</b> One of the strengths of a notebook is that you can move beyond static output. Below, you can explore category-specific word patterns and test your own review text live.
</div>

Run the cells below and try changing the controls. This is where programming starts to feel exploratory.

In [ ]:
print("Building interactive word explorer...")

def update_word_chart(selected_category="All", top_n=8):
    if selected_category == "All":
        filtered_df = reviews_df
    else:
        filtered_df = reviews_df[reviews_df["category"] == selected_category]
    
    print(f"Showing {len(filtered_df)} reviews from: {selected_category}")
    print(f"Average sentiment: {filtered_df['sentiment'].mean():.2f}")
    
    top_words = extract_top_words(filtered_df["cleaned_review"], top_n=top_n)
    
    if top_words.empty:
        print("No words available for this selection.")
        return
    
    fig = px.bar(
        top_words.sort_values("count"),
        x="count",
        y="word",
        orientation="h",
        color="count",
        color_continuous_scale="Tealgrn",
        template="plotly_white",
        title=f"Top {top_n} words for {selected_category}"
    )
    
    fig.update_layout(xaxis_title="Count", yaxis_title="")
    fig.show()

category_dropdown = widgets.Dropdown(
    options=["All"] + sorted(reviews_df["category"].unique().tolist()),
    value="All",
    description="Category:"
)

top_word_slider = widgets.IntSlider(
    value=8,
    min=5,
    max=12,
    step=1,
    description="Top words:",
    continuous_update=False
)

print("Widget ready! Explore the categories below:\n")
widgets.interact(update_word_chart, selected_category=category_dropdown, top_n=top_word_slider);

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try it yourself:</b> In the next cell, type your own short review. Then experiment by changing one or two words. For example, replace <i>great</i> with <i>frustrating</i>, or add an exclamation mark.
</div>

In [ ]:
print("Creating a live review analyser...")

user_review = widgets.Textarea(
    value="The playlists looked great and the app was easy to use, but the adverts became annoying.",
    placeholder="Type a short review here...",
    description="Your text:",
    layout=widgets.Layout(width="100%", height="120px"),
    continuous_update=False
)

analysis_output = widgets.Output()

def analyse_live_review(change=None):
    with analysis_output:
        clear_output(wait=True)
        text = user_review.value.strip()
        
        if not text:
            print("Type a short review to see the analysis.")
            return
        
        cleaned_text = clean_text(text)
        polarity = get_sentiment(text)
        label = get_sentiment_label(polarity)
        word_total = get_word_count(text)
        exclamation_total = text.count("!")
        
        summary_df = pd.DataFrame([
            {
                "Cleaned text": cleaned_text,
                "Word count": word_total,
                "Exclamation marks": exclamation_total,
                "Sentiment polarity": round(polarity, 3),
                "Sentiment label": label
            }
        ])
        
        display(summary_df)
        
        if polarity > 0.15:
            print("Interpretation: the language sounds broadly positive.")
        elif polarity < -0.15:
            print("Interpretation: the language sounds broadly negative.")
        else:
            print("Interpretation: the wording looks mixed or fairly neutral.")
        
        print("Tip: change one adjective and run the effect through again.")

user_review.observe(analyse_live_review, names="value")

display(user_review)
display(analysis_output)
analyse_live_review()

print("Live analyser ready!")

### Reading the Results Responsibly

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important:</b> These results are useful for learning, but they are not perfect truth. Responsible programming means understanding the limits of our methods as well as their strengths.
</div>

- <b>TextBlob sentiment is simple.</b> It gives a quick polarity estimate, but it can miss sarcasm, context, discipline-specific language, or mixed opinions.
- <b>Word clouds are eye-catching, not complete analysis.</b> They highlight frequent terms, but frequency alone does not explain meaning.
- <b>Cleaning choices matter.</b> Lowercasing text, removing punctuation, and dropping stopwords all affect what patterns appear.
- <b>Small datasets support small claims.</b> Our review set is intentionally compact for teaching, so we should not generalise too far from it.

This is a useful lesson beyond programming: good analysis is never only about getting an output. It is also about judging what that output can and cannot support.

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have built a complete mini workflow that moves from raw review text to cleaned data, derived features, sentiment estimates, visual outputs, interactivity, and debugging practice. That is a strong foundation for Introductory Python.
</div>

### Suggested next steps

- compare sentiment scores with star ratings more formally
- test different stopword lists and see how the top words change
- collect a larger dataset and explore patterns by time or source
- build your own function that flags keywords such as <i>slow</i>, <i>excellent</i>, or <i>confusing</i>

Programming becomes more powerful as soon as you start asking your own questions. Keep experimenting.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20Notebook%20Footer.png)